# exp_07 OOD evaluation — NCT-CRC-HE-7K colorectal dataset (extended)

OOD evaluation of LC25000-trained checkpoints on the **NCT-CRC-HE-7K** external colorectal pathology set (Kather et al., 2018, mirrored on Zenodo). Extended-variant version of `CLIP_OOD_eval.ipynb`: supports four model architectures via a `VARIANT` switch:

- `baseline` — `CLIPModel_HF` (ResNet50 image + BERT text + projection heads)
- `plain_classifier` — ResNet50 -> GAP -> BatchNorm -> Dense(5) softmax
- `plip_bert_composed` — `CLIPModel_PLIP` (PLIP ViT-B/32 + BERT + projection heads)
- `dinov2_bert_composed` — `CLIPModel_DINOv2` (DINOv2 ViT-B/14 + BERT + projection heads; exp_10 control)

The pre-existing `CLIP_OOD_eval.ipynb` is kept untouched so the archived `baseline` and `stain_macenko_nct_ref` runs remain reproducible. This extended notebook is the cleaner template going forward and adds the PLIP / DINOv2 backbones with the same `_CHECKPOINT_MAP` switch used by Chaoyang_OOD_eval and LungHist700_OOD_eval.

## Dataset

- Source: https://zenodo.org/records/1214456 (Zenodo mirror of Kather 2018)
- ~7K H&E 224×224 colorectal patches at 20× magnification
- 9 classes on disk: ADI, BACK, DEB, LYM, MUC, MUS, NORM, STR, TUM
- We restrict the OOD scoring to the **two classes with clean LC25000 analogues**:
    - `NORM` -> `benign colon tissue` (LC25000 idx 3)
    - `TUM`  -> `colon adenocarcinoma` (LC25000 idx 4)
- The other 7 classes (ADI/BACK/DEB/LYM/MUC/MUS/STR) are not scored — no LC25000 counterpart.

## Setup

NCT-CRC ships as a publicly-downloadable ~1.6 GB zip on Zenodo. Cell 3 caches it to Drive on first download so subsequent Colab sessions skip the network step. No license registration is needed.

**Citation:** Kather, J. N. et al. *100,000 histological images of human colorectal cancer and healthy tissue*. Zenodo (2018). DOI: 10.5281/zenodo.1214456.


<a href="https://colab.research.google.com/github/lcandau/histopathology-clip-lab/blob/exp_09_extended_ood_eval/experiments/exp_07_ood_eval/NCT_OOD_eval_extended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 0 -- Bootstrap


In [ ]:
# --- Cell 0: bootstrap ---
import os, sys, subprocess
os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/lcandau/histopathology-clip-lab.git"
REPO_DIR = "/content/histopathology-clip-lab"
BRANCH   = "exp_09_extended_ood_eval"
IN_COLAB = "google.colab" in sys.modules


def _clone_with_fallback(branch):
    try:
        subprocess.run(
            ["git", "clone", "-b", branch, REPO_URL, REPO_DIR],
            check=True, capture_output=True,
        )
        print(f"Cloned branch {branch!r}")
        return branch
    except subprocess.CalledProcessError:
        print(f"Branch {branch!r} not found; cloning main")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        return "main"


if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        active = _clone_with_fallback(BRANCH)
    else:
        fetch = subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True)
        if fetch.returncode == 0:
            subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
            active = BRANCH
        else:
            subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
            active = "main"
    print(f"Active branch: {active}")
    subprocess.run([
        "pip", "install", "-q",
        "tensorflow==2.18.0", "keras==3.7.0", "keras-hub==0.18.1",
        "transformers==4.46.0", "umap-learn", "kagglehub",
        "scikit-learn", "matplotlib", "pandas", "Pillow",
    ], check=True)
    subprocess.run(["pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
else:
    LOCAL_REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if LOCAL_REPO_DIR not in sys.path:
        sys.path.insert(0, LOCAL_REPO_DIR)

print("In Colab:", IN_COLAB)
print("sys.path[0]:", sys.path[0])
print("KERAS_BACKEND:", os.environ.get("KERAS_BACKEND"))


## 1 -- Imports


In [ ]:
# --- Cell 1: imports ---
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import json
import math
import shutil
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
import keras
import keras_hub

from transformers import AutoTokenizer, TFAutoModel, TFCLIPVisionModel
# NOTE: DINOv2 has no TF port in transformers==4.46.0 -- the dinov2_bert_composed
# variant consumes features precomputed by DINOv2_feature_precompute.ipynb.

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)

from src.utils.paths import is_colab, repo_root, drive_root, run_dir, results_dir
from src.utils.repro import set_global_seed, enable_op_determinism
from src.data.lc25000 import (
    CLASS_INFO, NUM_CLASSES, CLASS_NAMES, INDEX_TO_NAME, ID_TO_INDEX,
    discover_records, stratified_split, save_split, load_split,
)
from src.prompts.templates import baseline_prompt

## 2 -- Configuration (VARIANT switch lives here)


In [ ]:
# --- Cell 2: config ---
EXPERIMENT = "exp_07_ood_eval"
DATASET_TAG = "nct_crc"   # short tag used in output filenames

# --- VARIANT SWITCH (toggle and re-run from Cell 5 onwards) ---
# One of: "baseline", "plain_classifier", "plip_bert_composed", "dinov2_bert_composed"
VARIANT = "baseline"

SEED = 42
set_global_seed(SEED)
enable_op_determinism()

IMG_SIZE   = 224
MAX_LEN    = 24
EMBED_DIM  = 256
INIT_TEMP  = 0.07
BATCH_SIZE = 64

# --- LC25000 target classes for restricted argmax ---
# Each VARIANT lands in one of three families; the restricted-argmax indices
# are the same regardless of variant (CLIP-style cosine vs softmax logits both
# index the same 5-way LC25000 class space).
TARGET_LC_NAMES = [
    'benign colon tissue',
    'colon adenocarcinoma',
]
TARGET_LC_IDX = sorted({
    ID_TO_INDEX[next(c.id for c in CLASS_INFO if c.name == n)]
    for n in TARGET_LC_NAMES
})

# --- PLIP image normalisation constants (only used when VARIANT == plip_bert_composed) ---
PLIP_MODEL_ID   = "vinid/plip"
TEXT_ENCODER_ID = "bert-base-uncased"
PLIP_MEAN = np.array([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
PLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)

# --- DINOv2 precomputed-feature cache (only used when VARIANT == dinov2_bert_composed) ---
# Features are precomputed by experiments/exp_10_dinov2_backbone/DINOv2_feature_precompute.ipynb.
DINOV2_MODEL_ID = "facebook/dinov2-base"
DINOV2_VISION_HIDDEN = 768
DINOV2_CACHE_ROOT = Path("/content/drive/MyDrive/clip_histopathology/cache/dinov2")
DINOV2_OOD_CACHE_PATH     = DINOV2_CACHE_ROOT / f"{DATASET_TAG}.h5"
DINOV2_LC25000_CACHE_PATH = DINOV2_CACHE_ROOT / "lc25000.h5"

# --- Per-variant checkpoint paths (mirror the exp_07 OOD eval notebooks) ---
_CHECKPOINT_MAP = {
    "baseline":             run_dir("exp_01_baseline",      "baseline")             / "weights.weights.h5",
    "plain_classifier":     run_dir("exp_03_plain_classifier", "plain_classifier")  / "weights.weights.h5",
    "plip_bert_composed":   run_dir("exp_08_plip_backbone", "plip_bert_composed")   / "weights.weights.h5",
    "dinov2_bert_composed": run_dir("exp_10_dinov2_backbone", "dinov2_bert_composed") / "weights.weights.h5",
}
assert VARIANT in _CHECKPOINT_MAP, f"Unknown VARIANT {VARIANT!r}; pick one of {list(_CHECKPOINT_MAP)}"
CHECKPOINT_PATH = _CHECKPOINT_MAP[VARIANT]

METRICS_DIR = results_dir("metrics") / EXPERIMENT
CM_DIR      = results_dir("confusion_matrices") / EXPERIMENT
PLOTS_DIR   = results_dir("plots") / EXPERIMENT
for d in (METRICS_DIR, CM_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

OUT_TAG = f"{VARIANT}_{DATASET_TAG}"

print(f"Variant:        {VARIANT}")
print(f"Dataset tag:    {DATASET_TAG}")
print(f"Checkpoint:     {CHECKPOINT_PATH}")
print(f"Target classes: {TARGET_LC_NAMES}  -> LC25000 indices {TARGET_LC_IDX}")
print(f"Metrics out:    {METRICS_DIR}")
if VARIANT == "dinov2_bert_composed":
    print(f"DINOv2 cache:   {DINOV2_OOD_CACHE_PATH}")
    print(f"  + LC25000:    {DINOV2_LC25000_CACHE_PATH}")

## 3 -- Acquire NCT-CRC-HE-7K (Zenodo mirror + Drive cache)

In [ ]:
# --- Cell 3: download / extract NCT-CRC ---
import urllib.request, zipfile, shutil

NCT_URL  = "https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip"
NCT_NAME = "CRC-VAL-HE-7K"

DRIVE_OOD_ZIP   = drive_root() / "ood_datasets" / f"{NCT_NAME}.zip"
LOCAL_OOD_ROOT  = Path("/content/ood_data") if IN_COLAB else Path("/tmp/ood_data")
LOCAL_OOD_DIR   = LOCAL_OOD_ROOT / NCT_NAME


def _ensure_nct_zip(path):
    """Copy from Drive cache if present; otherwise download from Zenodo and persist to Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        print(f"NCT zip already at {path}")
        return path
    DRIVE_OOD_ZIP.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_OOD_ZIP.exists():
        print(f"Copying NCT zip from Drive cache {DRIVE_OOD_ZIP} -> {path}")
        shutil.copy(DRIVE_OOD_ZIP, path)
        return path
    print(f"Downloading NCT-CRC from {NCT_URL} ...")
    urllib.request.urlretrieve(NCT_URL, path)
    print(f"Copying download to Drive cache: {DRIVE_OOD_ZIP}")
    shutil.copy(path, DRIVE_OOD_ZIP)
    return path


nct_zip_local = LOCAL_OOD_ROOT / f"{NCT_NAME}.zip"
LOCAL_OOD_ROOT.mkdir(parents=True, exist_ok=True)
_ensure_nct_zip(nct_zip_local)

if not LOCAL_OOD_DIR.exists() or not any(LOCAL_OOD_DIR.iterdir()):
    print(f"Extracting {nct_zip_local} -> {LOCAL_OOD_ROOT}")
    with zipfile.ZipFile(nct_zip_local) as zf:
        zf.extractall(LOCAL_OOD_ROOT)

class_dirs = sorted(p for p in LOCAL_OOD_DIR.iterdir() if p.is_dir())
print(f"\nClasses on disk under {LOCAL_OOD_DIR}:")
for cd in class_dirs:
    n = sum(1 for _ in cd.iterdir())
    print(f"  {cd.name:>5s}  ({n} tiles)")


## 4 -- Build label arrays + LC25000 class mapping (NCT NORM/TUM only)

In [ ]:
# --- Cell 4: discover OOD records (NORM + TUM only) ---
# NCT folder name -> LC25000 prompt class name.
NCT_TO_LC25000 = {
    "NORM": "benign colon tissue",
    "TUM":  "colon adenocarcinoma",
}

ood_paths_list = []
ood_nct_labels_list = []
for nct_cls in NCT_TO_LC25000:
    class_dir = LOCAL_OOD_DIR / nct_cls
    if not class_dir.exists():
        raise FileNotFoundError(f"Expected NCT class dir not found: {class_dir}")
    files = sorted(class_dir.iterdir())
    for f in files:
        if f.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"}:
            ood_paths_list.append(str(f))
            ood_nct_labels_list.append(nct_cls)

ood_paths       = np.asarray(ood_paths_list)
ood_nct_labels  = np.asarray(ood_nct_labels_list)
ood_lc_indices  = np.array([
    ID_TO_INDEX[next(c.id for c in CLASS_INFO if c.name == NCT_TO_LC25000[lab])]
    for lab in ood_nct_labels
], dtype=np.int32)

# Keep a friendly alias for the UMAP cell — the Chaoyang notebook uses ood_chy_labels.
ood_chy_labels = ood_nct_labels  # kept for downstream symmetry with the Chaoyang cell layout

print(f"NCT-CRC-HE-7K OOD eval (NORM + TUM only):")
for nct_cls in NCT_TO_LC25000:
    n = int((ood_nct_labels == nct_cls).sum())
    print(f"  {nct_cls:>5s} -> {NCT_TO_LC25000[nct_cls]:>30s}: {n}")
print(f"\nTotal images: {len(ood_paths)}")
print(f"LC25000 target class indices: {TARGET_LC_IDX}")


## 5 -- Image loader (variant-gated preprocessing)


In [ ]:
# --- Cell: image loader (variant-gated) ---
# Three preprocessing paths:
#   - baseline / plain_classifier  -> /255.0 RGB in [0, 1]   (matches keras_hub resnet_50_imagenet)
#   - plip_bert_composed           -> /255.0 then CLIP mean/std (matches HF TFCLIPVisionModel)
#   - dinov2_bert_composed         -> features precomputed in PyTorch; image pixels not loaded here.


def load_pil_array(path):
    """Read a JPEG/PNG via PIL, resize to IMG_SIZE, return a float32 [0, 1] RGB array."""
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def preprocess_for_variant(arr01):
    """Apply variant-specific channel normalisation on top of a [0,1] RGB array."""
    if VARIANT == "plip_bert_composed":
        return (arr01 - PLIP_MEAN) / PLIP_STD
    # baseline + plain_classifier expect raw /255.0; dinov2 never calls this fn.
    return arr01


if VARIANT == "plip_bert_composed":
    _pre_label = "CLIP mean/std normalisation"
elif VARIANT == "dinov2_bert_composed":
    _pre_label = "(no per-image preprocessing -- features precomputed in PyTorch)"
else:
    _pre_label = "raw /255.0"
print(f"Preprocessing path for VARIANT={VARIANT}: {_pre_label}")

## 6 -- Model construction (variant-gated)


In [ ]:
# --- Cell: model construction (variant-gated) ---
# Each variant builds a model and assigns:
#   - encode_paths(paths) : (N, embed_dim) L2-normalised image embeddings
#   - score(image_emb)    : (N, 5) similarity/logit vector against the 5 LC25000 classes
#                            (cosine sim for CLIP variants; raw logits for plain_classifier)
# The downstream restricted-argmax cell then indexes score with TARGET_LC_IDX.

# We also expose a single `image_features(paths)` -> (N, F) "feature space" used
# by the shared UMAP. For CLIP variants, F = EMBED_DIM (projected unit vectors).
# For plain_classifier, F = 2048 (post-BN pooled ResNet features).


def _l2(x, axis=-1):
    return tf.math.l2_normalize(x, axis=axis)


if VARIANT in ("baseline", "plain_classifier"):
    image_backbone = keras_hub.models.Backbone.from_preset("resnet_50_imagenet")
    image_backbone.trainable = False

if VARIANT == "baseline":
    text_backbone = keras_hub.models.Backbone.from_preset("bert_base_en_uncased")
    text_backbone.trainable = False
    text_pre = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
        "bert_base_en_uncased", sequence_length=MAX_LEN,
    )

    class CLIPModel_HF(keras.Model):
        def __init__(self, img_backbone, text_backbone, embed_dim=EMBED_DIM, init_temp=INIT_TEMP, **kwargs):
            super().__init__(**kwargs)
            self.img_backbone  = img_backbone
            self.text_backbone = text_backbone
            self.img_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="img_projection")
            self.text_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="text_projection")
            self.logit_scale = self.add_weight(
                name="logit_scale", shape=(),
                initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
                trainable=True, dtype="float32",
            )
            self.loss_tracker = keras.metrics.Mean(name="loss")

        @property
        def metrics(self):
            return [self.loss_tracker]

        def encode_image(self, images, training=False):
            features = self.img_backbone(images, training=False)
            if features.shape.rank == 4:
                features = tf.reduce_mean(features, axis=[1, 2])
            return _l2(self.img_projection(features))

        def encode_text(self, token_dict, training=False):
            out = self.text_backbone(token_dict, training=False)
            if isinstance(out, dict):
                features = out.get("pooled_output", out.get("sequence_output"))
                if features is not None and features.shape.rank == 3:
                    features = features[:, 0, :]
            else:
                features = out[:, 0, :] if out.shape.rank == 3 else out
            return _l2(self.text_projection(features))

        def call(self, inputs, training=False):
            images, token_dict = inputs
            return self.encode_image(images, training=training), self.encode_text(token_dict, training=training)

    model = CLIPModel_HF(image_backbone, text_backbone)
    class_prompts = [baseline_prompt(name) for name in CLASS_NAMES]
    class_token_dict = text_pre(tf.constant(class_prompts))
    _ = model((tf.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
               {k: tf.constant(v[:1]) for k, v in class_token_dict.items()}),
              training=False)

elif VARIANT == "plain_classifier":
    inputs   = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    features = image_backbone(inputs, training=False)
    features = keras.layers.GlobalAveragePooling2D(name="gap")(features)
    features = keras.layers.BatchNormalization(name="feature_norm", dtype="float32")(features)
    logits   = keras.layers.Dense(NUM_CLASSES, name="classifier",
                                  kernel_initializer="glorot_uniform", dtype="float32")(features)
    model = keras.Model(inputs=inputs, outputs=[logits, features], name="plain_classifier_with_features")
    _ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32), training=False)

elif VARIANT == "plip_bert_composed":
    def _load_plip_vision():
        try:
            m = TFCLIPVisionModel.from_pretrained(PLIP_MODEL_ID, from_pt=False)
        except (OSError, EnvironmentError, TypeError, ValueError):
            m = TFCLIPVisionModel.from_pretrained(PLIP_MODEL_ID, from_pt=True)
        m.trainable = False
        return m

    def _load_bert_text(encoder_id=TEXT_ENCODER_ID):
        tokenizer = AutoTokenizer.from_pretrained(encoder_id)
        try:
            m = TFAutoModel.from_pretrained(encoder_id, from_pt=False)
        except (OSError, EnvironmentError, TypeError, ValueError):
            m = TFAutoModel.from_pretrained(encoder_id, from_pt=True)
        m.trainable = False
        return m, tokenizer, m.config.hidden_size

    print("Loading PLIP vision encoder...")
    plip_vision = _load_plip_vision()
    print("Loading BERT text encoder...")
    text_backbone, tokenizer, _text_hidden = _load_bert_text()

    def tokenize_prompts(tokenizer, prompts, max_len=MAX_LEN):
        out = tokenizer(prompts, padding="max_length", truncation=True,
                        max_length=max_len, return_tensors="tf")
        return {k: tf.cast(v, tf.int32) for k, v in out.items()}

    class CLIPModel_PLIP(keras.Model):
        def __init__(self, vision_model, text_backbone, embed_dim=EMBED_DIM,
                     init_temp=INIT_TEMP, **kwargs):
            super().__init__(**kwargs)
            self.vision_model  = vision_model
            self.text_backbone = text_backbone
            self.img_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="img_projection")
            self.text_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="text_projection")
            self.logit_scale = self.add_weight(
                name="logit_scale", shape=(),
                initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
                trainable=True, dtype="float32",
            )
            self.loss_tracker = keras.metrics.Mean(name="loss")

        @property
        def metrics(self):
            return [self.loss_tracker]

        def encode_image(self, pixel_values, training=False):
            # CRITICAL: HF's TFCLIPVisionEmbeddings expects NCHW (PyTorch convention).
            # tf.data feeds NHWC, so we transpose here.
            pixel_values = tf.transpose(pixel_values, perm=[0, 3, 1, 2])
            out = self.vision_model(pixel_values=pixel_values, training=False)
            features = out.pooler_output
            return _l2(self.img_projection(features))

        def encode_text(self, token_dict, training=False):
            kwargs = {"input_ids": token_dict["input_ids"],
                      "attention_mask": token_dict["attention_mask"]}
            if "token_type_ids" in token_dict:
                kwargs["token_type_ids"] = token_dict["token_type_ids"]
            out = self.text_backbone(**kwargs, training=False)
            pooled = getattr(out, "pooler_output", None)
            if pooled is None:
                pooled = out.last_hidden_state[:, 0, :]
            return _l2(self.text_projection(pooled))

        def call(self, inputs, training=False):
            pixel_values, token_dict = inputs
            return self.encode_image(pixel_values, training=training), self.encode_text(token_dict, training=training)

    model = CLIPModel_PLIP(plip_vision, text_backbone)
    class_prompts = [baseline_prompt(name) for name in CLASS_NAMES]
    class_token_dict = tokenize_prompts(tokenizer, class_prompts)
    _ = model((tf.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
               {k: tf.constant(v[:1]) for k, v in class_token_dict.items()}),
              training=False)

elif VARIANT == "dinov2_bert_composed":
    # DINOv2 has no TF port in transformers==4.46.0. Features are precomputed
    # in PyTorch by experiments/exp_10_dinov2_backbone/DINOv2_feature_precompute.ipynb
    # and loaded from HDF5 here. The Keras model is projection-head-only.
    for _p in (DINOV2_OOD_CACHE_PATH, DINOV2_LC25000_CACHE_PATH):
        if not _p.is_file():
            raise FileNotFoundError(
                f"DINOv2 feature cache missing: {_p}\n"
                "Run experiments/exp_10_dinov2_backbone/DINOv2_feature_precompute.ipynb first."
            )

    def _load_dinov2_cache(p):
        with h5py.File(p, "r") as f:
            feats = f["features"][:]
            keys  = [k.decode() if isinstance(k, bytes) else k for k in f["paths"][:]]
        return feats, {k: i for i, k in enumerate(keys)}

    _ood_feats, _ood_index = _load_dinov2_cache(DINOV2_OOD_CACHE_PATH)
    _lc_feats,  _lc_index  = _load_dinov2_cache(DINOV2_LC25000_CACHE_PATH)
    DINOV2_OOD_ROOT = LOCAL_OOD_DIR   # filled per notebook
    _LC_TOKEN = "lung_colon_image_set"

    def _dinov2_lookup(path):
        """Resolve a (LC25000 or OOD-dataset) absolute path to its 768-d feature."""
        p = Path(path).as_posix()
        if f"/{_LC_TOKEN}/" in p:
            key = p.split(f"/{_LC_TOKEN}/", 1)[1]
            idx = _lc_index.get(key)
            if idx is not None:
                return _lc_feats[idx]
        try:
            rel = Path(path).resolve().relative_to(Path(DINOV2_OOD_ROOT).resolve()).as_posix()
        except ValueError:
            rel = Path(path).name
        idx = _ood_index.get(rel)
        if idx is None:
            # LungHist700 cache keys are bare basenames; try that.
            idx = _ood_index.get(Path(path).name)
        if idx is None:
            raise KeyError(
                f"DINOv2 cache miss for {path!r} (looked up rel={rel!r}). "
                "Re-run DINOv2_feature_precompute.ipynb."
            )
        return _ood_feats[idx]

    def _load_bert_text(encoder_id=TEXT_ENCODER_ID):
        tokenizer = AutoTokenizer.from_pretrained(encoder_id)
        try:
            m = TFAutoModel.from_pretrained(encoder_id, from_pt=False)
        except (OSError, EnvironmentError, TypeError, ValueError):
            m = TFAutoModel.from_pretrained(encoder_id, from_pt=True)
        m.trainable = False
        return m, tokenizer, m.config.hidden_size

    print(f"Loaded DINOv2 caches: OOD={_ood_feats.shape}  LC25000={_lc_feats.shape}")
    print("Loading BERT text encoder...")
    text_backbone, tokenizer, _text_hidden = _load_bert_text()

    def tokenize_prompts(tokenizer, prompts, max_len=MAX_LEN):
        out = tokenizer(prompts, padding="max_length", truncation=True,
                        max_length=max_len, return_tensors="tf")
        return {k: tf.cast(v, tf.int32) for k, v in out.items()}

    class CLIPModel_DINOv2(keras.Model):
        """Projection-head CLIP over precomputed DINOv2 features."""
        def __init__(self, text_backbone, embed_dim=EMBED_DIM,
                     init_temp=INIT_TEMP, **kwargs):
            super().__init__(**kwargs)
            self.text_backbone = text_backbone
            self.img_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="img_projection")
            self.text_projection = keras.Sequential([
                keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
                keras.layers.LayerNormalization(dtype="float32"),
            ], name="text_projection")
            self.logit_scale = self.add_weight(
                name="logit_scale", shape=(),
                initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
                trainable=True, dtype="float32",
            )
            self.loss_tracker = keras.metrics.Mean(name="loss")

        @property
        def metrics(self):
            return [self.loss_tracker]

        def encode_image(self, vision_features, training=False):
            return _l2(self.img_projection(vision_features))

        def encode_text(self, token_dict, training=False):
            kwargs = {"input_ids": token_dict["input_ids"],
                      "attention_mask": token_dict["attention_mask"]}
            if "token_type_ids" in token_dict:
                kwargs["token_type_ids"] = token_dict["token_type_ids"]
            out = self.text_backbone(**kwargs, training=False)
            pooled = getattr(out, "pooler_output", None)
            if pooled is None:
                pooled = out.last_hidden_state[:, 0, :]
            return _l2(self.text_projection(pooled))

        def call(self, inputs, training=False):
            vision_features, token_dict = inputs
            return self.encode_image(vision_features, training=training), self.encode_text(token_dict, training=training)

    model = CLIPModel_DINOv2(text_backbone)
    class_prompts = [baseline_prompt(name) for name in CLASS_NAMES]
    class_token_dict = tokenize_prompts(tokenizer, class_prompts)
    _ = model((tf.zeros((1, DINOV2_VISION_HIDDEN), dtype=tf.float32),
               {k: tf.constant(v[:1]) for k, v in class_token_dict.items()}),
              training=False)


n_trainable = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"VARIANT={VARIANT}  trainable params: {n_trainable:,}")


## 7 -- Load checkpoint


In [ ]:
# --- Cell: load checkpoint ---
assert CHECKPOINT_PATH.is_file(), (
    f"Missing checkpoint at {CHECKPOINT_PATH}.\n"
    f"Run the training notebook for VARIANT={VARIANT!r} first and make sure the\n"
    f"weights file is in Drive at the expected path."
)
model.load_weights(str(CHECKPOINT_PATH))
print(f"Loaded weights from {CHECKPOINT_PATH}")


## 8 -- Scoring / feature helpers


In [ ]:
# --- Cell: scoring / feature extraction helpers ---
# Encoders for both classification scores (against the 5 LC25000 classes) and
# UMAP features (256-d for CLIP variants, 2048-d for plain_classifier).

if VARIANT == "baseline":
    prompt_embeddings_all = model.encode_text(class_token_dict, training=False).numpy()  # (5, 256)

    def encode_paths(paths, batch_size=BATCH_SIZE):
        scores, feats = [], []
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            batch_arrs = np.stack([preprocess_for_variant(load_pil_array(p)) for p in batch_paths])
            emb = model.encode_image(tf.constant(batch_arrs), training=False).numpy()
            feats.append(emb)
            scores.append(emb @ prompt_embeddings_all.T)
        return np.concatenate(scores, axis=0), np.concatenate(feats, axis=0)

elif VARIANT == "plip_bert_composed":
    prompt_embeddings_all = model.encode_text(class_token_dict, training=False).numpy()  # (5, 256)

    def encode_paths(paths, batch_size=BATCH_SIZE):
        scores, feats = [], []
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            batch_arrs = np.stack([preprocess_for_variant(load_pil_array(p)) for p in batch_paths])
            emb = model.encode_image(tf.constant(batch_arrs), training=False).numpy()
            feats.append(emb)
            scores.append(emb @ prompt_embeddings_all.T)
        return np.concatenate(scores, axis=0), np.concatenate(feats, axis=0)

elif VARIANT == "dinov2_bert_composed":
    prompt_embeddings_all = model.encode_text(class_token_dict, training=False).numpy()  # (5, 256)

    def encode_paths(paths, batch_size=BATCH_SIZE):
        # No image decoding: features were precomputed by the DINOv2 precompute
        # notebook. We gather them via _dinov2_lookup (handles both OOD-dataset
        # paths and LC25000 reference paths) and run them through the projection
        # head batched.
        scores, feats = [], []
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            batch_vis = np.stack([_dinov2_lookup(p) for p in batch_paths]).astype(np.float32)
            emb = model.encode_image(tf.constant(batch_vis), training=False).numpy()
            feats.append(emb)
            scores.append(emb @ prompt_embeddings_all.T)
        return np.concatenate(scores, axis=0), np.concatenate(feats, axis=0)

elif VARIANT == "plain_classifier":
    # No prompt embeddings; the Dense(5) weights serve as "class directions" for UMAP.
    classifier_layer = model.get_layer("classifier")
    W, _b = classifier_layer.get_weights()       # W: (2048, 5)
    class_directions = W.T                       # (5, 2048)
    prompt_embeddings_all = None                 # signals: no prompt vectors

    def encode_paths(paths, batch_size=BATCH_SIZE):
        scores, feats = [], []
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            batch_arrs = np.stack([preprocess_for_variant(load_pil_array(p)) for p in batch_paths])
            lo, fe = model(tf.constant(batch_arrs), training=False)
            scores.append(lo.numpy())
            feats.append(fe.numpy())
        return np.concatenate(scores, axis=0), np.concatenate(feats, axis=0)


print(f"encode_paths configured for VARIANT={VARIANT}")


## 9 -- OOD inference + metrics

Restricted argmax over the two colon classes (`benign colon tissue`, `colon adenocarcinoma`). For CLIP variants this is `argmax_{c in COLON_IDX} cosine(image, prompt_c)`; for `plain_classifier` it is `argmax_{c in COLON_IDX} logit_c`.


In [ ]:
# --- Cell: OOD inference + restricted argmax + metrics ---
print(f"Encoding {len(ood_paths)} OOD images...")
ood_scores, ood_features = encode_paths(ood_paths)
print(f"OOD scores:   {ood_scores.shape}")
print(f"OOD features: {ood_features.shape}")

# Restrict argmax to the target LC classes for this dataset.
scores_target = ood_scores[:, TARGET_LC_IDX]
pred_within = scores_target.argmax(axis=1)
y_pred = np.array([TARGET_LC_IDX[i] for i in pred_within], dtype=np.int32)
y_true = ood_lc_indices

print(f"y_pred unique: {sorted(set(y_pred.tolist()))}")
print(f"y_true unique: {sorted(set(y_true.tolist()))}")

labels = TARGET_LC_IDX
target_names = [INDEX_TO_NAME[i] for i in labels]

acc      = accuracy_score(y_true, y_pred)
bal      = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
wgt_f1   = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

print(f"\nOOD ({VARIANT} -> {DATASET_TAG}, restricted to {target_names})\n")
print(f"  accuracy          = {acc:.4f}")
print(f"  balanced accuracy = {bal:.4f}")
print(f"  macro F1          = {macro_f1:.4f}")
print(f"  weighted F1       = {wgt_f1:.4f}")

per_class = classification_report(
    y_true, y_pred, labels=labels, target_names=target_names,
    output_dict=True, zero_division=0,
)
print("\nPer-class:")
print(pd.DataFrame(per_class).T.round(4).to_string())

cm_raw = confusion_matrix(y_true, y_pred, labels=labels)
row_sums = cm_raw.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_raw, row_sums, where=row_sums > 0,
                    out=np.zeros_like(cm_raw, dtype=np.float64))

metrics_payload = {
    "experiment":          EXPERIMENT,
    "variant":             VARIANT,
    "dataset":             DATASET_TAG,
    "checkpoint":          str(CHECKPOINT_PATH),
    "n_images":            int(len(ood_paths)),
    "accuracy":            float(acc),
    "balanced_accuracy":   float(bal),
    "macro_f1":            float(macro_f1),
    "weighted_f1":         float(wgt_f1),
    "per_class":           per_class,
    "target_lc_indices":   labels,
    "target_names":        target_names,
}
metrics_path = METRICS_DIR / f"{OUT_TAG}_ood_classification.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2))
np.save(CM_DIR / f"{OUT_TAG}_ood_cm_raw.npy",  cm_raw)
np.save(CM_DIR / f"{OUT_TAG}_ood_cm_norm.npy", cm_norm)
print(f"\nSaved {metrics_path}")


## 10 -- Confusion-matrix plot


In [ ]:
# --- Cell: confusion-matrix plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mat, title, fmt, cmap in [
    (axes[0], cm_raw,  "Confusion matrix (raw)",            "d",   "Blues"),
    (axes[1], cm_norm, "Confusion matrix (row-normalised)", ".2f", "Blues"),
]:
    im = ax.imshow(mat, cmap=cmap)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(target_names, rotation=20, ha="right", fontsize=9)
    ax.set_yticklabels(target_names, fontsize=9)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() * 0.6 else "black", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(f"OOD CM — {VARIANT} / {DATASET_TAG}", fontsize=11)
plt.tight_layout()
cm_path = PLOTS_DIR / f"{OUT_TAG}_ood_cm.png"
fig.savefig(cm_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved {cm_path}")


## 11 -- LC25000 reference + shared UMAP fit


In [ ]:
# --- Cell: LC25000 reference (in-distribution embeddings) ---
import umap
import kagglehub

LC_SPLIT_PATH = results_dir("splits") / f"lc25000_seed{SEED}.json"
if LC_SPLIT_PATH.exists():
    lc_split = load_split(LC_SPLIT_PATH)
    print(f"Loaded LC25000 split from {LC_SPLIT_PATH}")
else:
    print("Downloading LC25000 via kagglehub...")
    lc_dataset_path = kagglehub.dataset_download(
        "andrewmvd/lung-and-colon-cancer-histopathological-images"
    )
    all_paths, all_indices = discover_records(lc_dataset_path)
    lc_split = stratified_split(all_paths, all_indices,
                                val_fraction=0.10, test_fraction=0.10, seed=SEED)
    save_split(lc_split, LC_SPLIT_PATH)

lc_test_paths   = np.asarray(lc_split["test_paths"])
lc_test_indices = np.asarray(lc_split["test_indices"], dtype=np.int32)
print(f"LC25000 test set: {len(lc_test_paths)} images")
for idx in range(len(CLASS_INFO)):
    print(f"  [{idx}] {INDEX_TO_NAME[idx]}: {int((lc_test_indices == idx).sum())}")

lc_target_mask    = np.isin(lc_test_indices, TARGET_LC_IDX)
lc_target_paths   = lc_test_paths[lc_target_mask]
lc_target_indices = lc_test_indices[lc_target_mask]

print(f"\nEncoding {len(lc_test_paths)} LC25000 test images (with VARIANT={VARIANT} preprocessing)...")
_lc_scores, lc_test_features = encode_paths(lc_test_paths)
print(f"LC25000 features: {lc_test_features.shape}")

# Compose vectors for the UMAP: image features + 5 "anchor" vectors (prompts or class directions).
if VARIANT == "plain_classifier":
    anchor_vectors = class_directions
    anchor_kind = "class-direction"
else:
    anchor_vectors = prompt_embeddings_all
    anchor_kind = "prompt"
print(f"Anchor vectors ({anchor_kind}): {anchor_vectors.shape}")

print(f"\nFitting combined UMAP over {len(lc_test_features) + len(ood_features) + len(anchor_vectors)} points...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
combined = np.vstack([lc_test_features, ood_features, anchor_vectors])
combined_2d = reducer.fit_transform(combined)

n_lc, n_ood = len(lc_test_features), len(ood_features)
lc_2d         = combined_2d[:n_lc]
ood_2d        = combined_2d[n_lc:n_lc + n_ood]
anchor_2d     = combined_2d[n_lc + n_ood:]
lc_target_2d  = lc_2d[lc_target_mask]
anchor_2d_tgt = anchor_2d[TARGET_LC_IDX]
print(f"  lc_2d={lc_2d.shape}  ood_2d={ood_2d.shape}  anchor_2d={anchor_2d.shape}")


## 12 -- 2x3 UMAP grid (scatter + thumbnails)


In [ ]:
# --- Cell: 2x3 UMAP grid (scatter + thumbnails) ---
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch

CLASS_COLOR = {
    0: "#2ca02c",  # benign lung tissue           - green
    1: "#ff7f0e",  # lung adenocarcinoma          - orange
    2: "#9467bd",  # lung squamous cell carcinoma - purple
    3: "#08519c",  # benign colon tissue          - blue
    4: "#a50f15",  # colon adenocarcinoma         - red
}

OOD_LABEL = "NCT-CRC"


def draw_anchors(ax, anchor_pos, indices, *, with_labels=False, size=260):
    label_prefix = "class" if VARIANT == "plain_classifier" else "prompt"
    for k, idx in enumerate(indices):
        color = CLASS_COLOR[idx]
        ax.scatter(anchor_pos[k, 0], anchor_pos[k, 1], s=size, marker="*",
                   c=color, edgecolors="white", linewidths=2.0, zorder=6)
        if with_labels:
            ax.annotate(f"{label_prefix}:\n{INDEX_TO_NAME[idx]}",
                        (anchor_pos[k, 0], anchor_pos[k, 1]),
                        xytext=(12, 10), textcoords="offset points",
                        fontsize=8, fontweight="bold", zorder=7,
                        bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                                  edgecolor=color, alpha=0.9, linewidth=1.5))


def _scatter_ood(ax, alpha=0.55, size=10):
    """Scatter OOD points colour-coded by their mapped LC25000 class index."""
    for idx in sorted(set(ood_lc_indices.tolist())):
        mask = (ood_lc_indices == idx)
        ax.scatter(ood_2d[mask, 0], ood_2d[mask, 1], s=size, alpha=alpha,
                   c=CLASS_COLOR[idx], marker="^",
                   label=f"{OOD_LABEL} {INDEX_TO_NAME[idx]}",
                   edgecolors="white", linewidths=0.3)


def _read_thumb(path, size=(56, 56)):
    try:
        return np.asarray(Image.open(path).convert("RGB").resize(size, Image.BILINEAR))
    except Exception:
        return np.full((size[1], size[0], 3), 200, dtype=np.uint8)


def _layout_thumbnails(ax, coords, paths, class_colors, *, max_thumbs=35,
                       thumb_size=(56, 56), min_dist_frac=0.06, seed=42, border_width=2.4):
    coords = np.asarray(coords)
    rng = np.random.default_rng(seed)
    x_range = float(coords[:, 0].max() - coords[:, 0].min())
    y_range = float(coords[:, 1].max() - coords[:, 1].min())
    min_dist = min_dist_frac * min(x_range, y_range)
    idx = np.arange(len(coords)); rng.shuffle(idx)
    placed = []
    for i in idx:
        x, y = float(coords[i, 0]), float(coords[i, 1])
        if any((x - px) ** 2 + (y - py) ** 2 < min_dist ** 2 for (px, py) in placed):
            continue
        thumb = _read_thumb(paths[i], size=thumb_size)
        ab = AnnotationBbox(OffsetImage(thumb, zoom=1.0), (x, y),
                            frameon=True, pad=0.22,
                            bboxprops=dict(linewidth=border_width, edgecolor=class_colors[i]))
        ax.add_artist(ab)
        placed.append((x, y))
        if len(placed) >= max_thumbs:
            break
    return len(placed)


fig, axes = plt.subplots(2, 3, figsize=(20, 13))

x_min = min(lc_2d[:, 0].min(), ood_2d[:, 0].min(), anchor_2d[:, 0].min())
x_max = max(lc_2d[:, 0].max(), ood_2d[:, 0].max(), anchor_2d[:, 0].max())
y_min = min(lc_2d[:, 1].min(), ood_2d[:, 1].min(), anchor_2d[:, 1].min())
y_max = max(lc_2d[:, 1].max(), ood_2d[:, 1].max(), anchor_2d[:, 1].max())
pad_x = 0.04 * (x_max - x_min); pad_y = 0.04 * (y_max - y_min)
XLIM = (x_min - pad_x, x_max + pad_x); YLIM = (y_min - pad_y, y_max + pad_y)


def _box(ax, title):
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)


# ROW 1: SCATTER
ax = axes[0, 0]
for idx in range(len(CLASS_INFO)):
    mask = (lc_test_indices == idx)
    ax.scatter(lc_2d[mask, 0], lc_2d[mask, 1], s=8, alpha=0.5,
               c=CLASS_COLOR[idx], marker="o", label=f"LC25000 {INDEX_TO_NAME[idx]}", edgecolors="none")
_scatter_ood(ax, alpha=0.6, size=12)
draw_anchors(ax, anchor_2d, list(range(len(CLASS_INFO))))
ax.legend(loc="best", fontsize=6, framealpha=0.85)
_box(ax, f"LC25000 all 5 (circles) + {OOD_LABEL} (triangles)")

ax = axes[0, 1]
for idx in TARGET_LC_IDX:
    mask = (lc_test_indices == idx)
    ax.scatter(lc_2d[mask, 0], lc_2d[mask, 1], s=8, alpha=0.5,
               c=CLASS_COLOR[idx], marker="o", label=f"LC25000 {INDEX_TO_NAME[idx]}", edgecolors="none")
_scatter_ood(ax, alpha=0.6, size=12)
draw_anchors(ax, anchor_2d_tgt, TARGET_LC_IDX)
ax.legend(loc="best", fontsize=7, framealpha=0.85)
_box(ax, f"LC25000 target subset (circles) + {OOD_LABEL} (triangles)")

ax = axes[0, 2]
_scatter_ood(ax, alpha=0.6, size=10)
draw_anchors(ax, anchor_2d_tgt, TARGET_LC_IDX)
ax.legend(loc="best", fontsize=7, framealpha=0.85)
_box(ax, f"{OOD_LABEL} only  (F1={macro_f1:.3f})")


# ROW 2: THUMBNAILS
ax = axes[1, 0]
for idx in range(len(CLASS_INFO)):
    mask = (lc_test_indices == idx)
    ax.scatter(lc_2d[mask, 0], lc_2d[mask, 1], s=5, alpha=0.12, c=CLASS_COLOR[idx], marker="o")
_scatter_ood(ax, alpha=0.18, size=5)
_layout_thumbnails(ax, lc_2d, list(lc_test_paths),
    [CLASS_COLOR[int(lc_test_indices[i])] for i in range(len(lc_test_paths))],
    max_thumbs=25, min_dist_frac=0.07, seed=SEED, border_width=2.4)
_layout_thumbnails(ax, ood_2d, list(ood_paths),
    [CLASS_COLOR[int(ood_lc_indices[i])] for i in range(len(ood_paths))],
    max_thumbs=15, min_dist_frac=0.07, seed=SEED + 1, border_width=4.0)
draw_anchors(ax, anchor_2d, list(range(len(CLASS_INFO))))
_box(ax, f"LC25000 all + {OOD_LABEL} (thin / thick border)")

ax = axes[1, 1]
for idx in TARGET_LC_IDX:
    mask = (lc_test_indices == idx)
    ax.scatter(lc_2d[mask, 0], lc_2d[mask, 1], s=5, alpha=0.12, c=CLASS_COLOR[idx], marker="o")
_scatter_ood(ax, alpha=0.18, size=5)
_layout_thumbnails(ax, lc_target_2d, list(lc_target_paths),
    [CLASS_COLOR[int(lc_target_indices[i])] for i in range(len(lc_target_paths))],
    max_thumbs=20, min_dist_frac=0.07, seed=SEED, border_width=2.4)
_layout_thumbnails(ax, ood_2d, list(ood_paths),
    [CLASS_COLOR[int(ood_lc_indices[i])] for i in range(len(ood_paths))],
    max_thumbs=15, min_dist_frac=0.07, seed=SEED + 1, border_width=4.0)
draw_anchors(ax, anchor_2d_tgt, TARGET_LC_IDX)
_box(ax, f"LC25000 target + {OOD_LABEL} (thin / thick border)")

ax = axes[1, 2]
_scatter_ood(ax, alpha=0.20, size=5)
_layout_thumbnails(ax, ood_2d, list(ood_paths),
    [CLASS_COLOR[int(ood_lc_indices[i])] for i in range(len(ood_paths))],
    max_thumbs=30, min_dist_frac=0.06, seed=SEED, border_width=2.4)
draw_anchors(ax, anchor_2d_tgt, TARGET_LC_IDX)
_box(ax, f"{OOD_LABEL} only (thumbnails)")


fig.suptitle(f"LC25000 all + {OOD_LABEL}  ·  LC25000 target + {OOD_LABEL}  ·  {OOD_LABEL} only "
             f"-- shared UMAP -- variant: {VARIANT}", fontsize=14, y=1.005)
plt.tight_layout()
p_grid = PLOTS_DIR / f"{OUT_TAG}_ood_umap_grid.png"
fig.savefig(p_grid, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved {p_grid}")


## Summary

Run-time output (filenames include the active `VARIANT` and dataset tag):
- `results/metrics/exp_07_ood_eval/{VARIANT}_nct_crc_ood_classification.json`
- `results/confusion_matrices/exp_07_ood_eval/{VARIANT}_nct_crc_ood_cm_{raw,norm}.npy`
- `results/plots/exp_07_ood_eval/{VARIANT}_nct_crc_ood_cm.png`
- `results/plots/exp_07_ood_eval/{VARIANT}_nct_crc_ood_umap_grid.png`

**To run a different variant**, change `VARIANT` in Cell 2 and re-execute from the model-construction cell onwards (the bootstrap, imports, config, and dataset cells are variant-agnostic). Each variant writes its own files so re-runs are non-destructive.

**Prereq:** the training notebook for the selected VARIANT must have been run first to produce `<RUN_DIR>/weights.weights.h5` on Drive. See the run paths printed by Cell 2.
